## Project Description

This is an Abstract Syntax Tree (AST) analysis project created by Khalid Mihlar.

### Project Goal

I am analyzing student submissions from Assignment 3 of the CS 2420 class. All student submissions are de-identified and anonymous. The primary goal is to examine how the `size` variable, located within the `ArrayCollection` function (which extends `Collection` in Java), is interacted with, updated, and mutated across different functions. I will be creating an AST and analyzing each node to identify these interactions.


## Imports and Such
The main goal fo this block is to load in the proper location for the student submissions and then simply confirm what files were found and the name of the files

In [54]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import javalang
import pandas as pd

# Set this to the folder that contains your .java files.
# "." means the same folder as the notebook's current working directory.
ROOT_DIR = Path("./inputs/submissions")
JAVA_GLOB = "*.java"

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Folder does not exist: {ROOT_DIR.resolve()}")

java_files = sorted(ROOT_DIR.glob(JAVA_GLOB))

print(f"Looking for Java files in: {ROOT_DIR.resolve()}")
print("Found files:")
for p in java_files:
    print(" -", p.name)

if len(java_files) == 0:
    print("\nNo .java files found.")
    print("Put your five selected Java files in the folder above,")
    print("or change ROOT_DIR to the correct folder path.")
else:
    print(f"\nConfirmed {len(java_files)} Java file(s) found.")

Looking for Java files in: /Users/khalidmihlar/Code/ast-project/inputs/submissions
Found files:
 - AfricanPenguin.java
 - AfricanWildDog.java
 - Amphibia.java
 - Anthozoa.java
 - ArcticWolf.java
 - Armadillo.java
 - AsianGiantHornet.java
 - AsiaticBlackBear.java
 - Baboon.java
 - Bandicoot.java
 - Bivalvia.java
 - BlackRhinoceros.java
 - Bobcat.java
 - BorneoElephant.java
 - Bullfrog.java
 - BumbleBee.java
 - Caterpillar.java
 - Catfish.java
 - Centipede.java
 - Chicken.java
 - Cichlid.java
 - Cow.java
 - Crab.java
 - CrabEatingMacaque.java
 - Crane.java
 - CrestedPenguin.java
 - CrossRiverGorilla.java
 - Crustacea.java
 - Cuttlefish.java
 - Deer.java
 - Dhole.java
 - Discus.java
 - Dolphin.java
 - Dormouse.java
 - Dugong.java
 - FlyingSquirrel.java
 - Frigatebird.java
 - FrilledLizard.java
 - Goose.java
 - Gorilla.java
 - Grasshopper.java
 - GreenBeeEater.java
 - GreyReefShark.java
 - Hamster.java
 - Hare.java
 - HighlandCattle.java
 - HoneyBee.java
 - HornedFrog.java
 - Impala.java
 

## This section is focused on taking the files found in a folder previously and then attempt to see if all the files can be parsed into a tree and saved into raw_trees


In [55]:
def load_source(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def parse_java_file(path: Path):
    src = load_source(path)
    return javalang.parse.parse(src)

raw_trees = {}
parse_errors = {}

for path in java_files:
    try:
        raw_trees[path.name] = parse_java_file(path)
        print(f"Parsed successfully: {path.name}")
    except Exception as e:
        parse_errors[path.name] = str(e)
        print(f"Failed to parse: {path.name}")
        print(f"Error: {e}")
        print("-" * 50)



Parsed successfully: AfricanPenguin.java
Parsed successfully: AfricanWildDog.java
Parsed successfully: Amphibia.java
Parsed successfully: Anthozoa.java
Parsed successfully: ArcticWolf.java
Parsed successfully: Armadillo.java
Parsed successfully: AsianGiantHornet.java
Parsed successfully: AsiaticBlackBear.java
Parsed successfully: Baboon.java
Parsed successfully: Bandicoot.java
Parsed successfully: Bivalvia.java
Parsed successfully: BlackRhinoceros.java
Parsed successfully: Bobcat.java
Parsed successfully: BorneoElephant.java
Parsed successfully: Bullfrog.java
Parsed successfully: BumbleBee.java
Parsed successfully: Caterpillar.java
Parsed successfully: Catfish.java
Parsed successfully: Centipede.java
Parsed successfully: Chicken.java
Parsed successfully: Cichlid.java
Parsed successfully: Cow.java
Parsed successfully: Crab.java
Parsed successfully: CrabEatingMacaque.java
Parsed successfully: Crane.java
Parsed successfully: CrestedPenguin.java
Parsed successfully: CrossRiverGorilla.java


# Leftover Code
This was here for some understanding, don't want to delete it just in case I need it, but it had functionality for me to understand what was going on with some of the nodes

In [13]:
import javalang
import json

def dump_nodes(tree):
    for path, node in tree:
        print("=" * 60)
        print("NODE TYPE:", type(node).__name__)
        print("ATTRS:", getattr(node, "attrs", []))

        for attr in getattr(node, "attrs", []):
            print(f"  {attr}: {getattr(node, attr)}")

# Simply a way for me to see all the nodes and it's relevant information to get a better understanding 
def visualize_ast(node, indent=0):
    prefix = "  " * indent

    if isinstance(node, javalang.ast.Node):
        print(f"{prefix}{type(node).__name__}")

        for attr in node.attrs:
            value = getattr(node, attr)
            if value is None or value == []:
                continue

            print(f"{prefix}  .{attr}:")
            visualize_ast(value, indent + 2)

    elif isinstance(node, list):
        print(f"{prefix}list[{len(node)}]")
        for i, item in enumerate(node):
            print(f"{prefix}  [{i}]")
            visualize_ast(item, indent + 2)

    else:
        print(f"{prefix}{repr(node)}")

java_code = """
public class Example {
    public int test() {
        int size = 0;
        size++;
        return size;
    }
}
"""

print(raw_trees.keys())


dict_keys(['AfricanWildDog.java', 'Anthozoa.java', 'BlackRhinoceros.java', 'Newt.java', 'Yak.java'])


## Dataclass
This is where the AnalyzeRow data structure is defined for .csv population at the end


In [57]:
from dataclasses import dataclass, field

@dataclass
class AnalysisRow:
    file_name: str
    class_name: str
    method_name: str
    kind: str
    line_numbers: list[int] = field(default_factory=list)
    tags: list[str] = field(default_factory=list)
    metadata: str = ""

## Size and Constructor Helper Functions
Simple Ones

analyze_size_function_node

analyze_constructor_function_node

In [58]:
# public int size()
def analyze_size_function_node(node, file_name):
    tags = []
    line_numbers = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    if len(node.body) == 1:
        stmt = node.body[0]

        if isinstance(stmt, javalang.tree.ReturnStatement):
            if stmt.position is not None:
                line_numbers.append(stmt.position.line)

            expr = stmt.expression

            if isinstance(expr, javalang.tree.MemberReference):
                if expr.member == "size":
                    tags.append("returns_size")

            elif isinstance(expr, javalang.tree.This):
                tags.append("returns_this_size")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="size",
        kind="method",
        line_numbers=line_numbers,
        tags=tags,
        metadata=str(node)
    )

def analyze_constructor_function_node(node, file_name):
    tags = []
    line_numbers = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    size_set_to_zero = False

    for path, inner_node in node:
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl
            right_side = inner_node.value

            if (
                isinstance(left_side, javalang.tree.MemberReference)
                and left_side.member == "size"
                and isinstance(right_side, javalang.tree.Literal)
                and right_side.value == "0"
            ):
                size_set_to_zero = True

                if inner_node.position is not None:
                    line_numbers.append(inner_node.position.line)

    if size_set_to_zero:
        tags.append("sets_size_to_0")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="ArrayCollection",
        kind="constructor",
        line_numbers=sorted(set(line_numbers)),
        tags=tags,
        metadata=str(node)
    )

## Add function helper and stuff


In [59]:
def contains_size(expr):
    if isinstance(expr, javalang.tree.MemberReference):
        return expr.member == "size"
    if isinstance(expr, javalang.tree.BinaryOperation):
        return contains_size(expr.operandl) or contains_size(expr.operandr)
    return False


def contains_length(expr):
    if isinstance(expr, javalang.tree.MemberReference):
        return expr.member == "length"
    if isinstance(expr, javalang.tree.BinaryOperation):
        return contains_length(expr.operandl) or contains_length(expr.operandr)
    return False


def is_size_plus_one(expr):
    return (
        isinstance(expr, javalang.tree.BinaryOperation)
        and expr.operator == "+"
        and (
            (
                isinstance(expr.operandl, javalang.tree.MemberReference)
                and expr.operandl.member == "size"
                and isinstance(expr.operandr, javalang.tree.Literal)
                and expr.operandr.value == "1"
            )
            or
            (
                isinstance(expr.operandr, javalang.tree.MemberReference)
                and expr.operandr.member == "size"
                and isinstance(expr.operandl, javalang.tree.Literal)
                and expr.operandl.value == "1"
            )
        )
    )

def contains_size_function_call(expr):
    if isinstance(expr, javalang.tree.MethodInvocation):
        return expr.member == "size" and len(expr.arguments) == 0
    if isinstance(expr, javalang.tree.BinaryOperation):
        return contains_size_function_call(expr.operandl) or contains_size_function_call(expr.operandr)
    return False


def analyze_add_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    grow_called = False
    grow_called_with_capacity_check = False

    size_updated = False
    standard_increment = False
    nonstandard_size_change = False

    array_insert_uses_size = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # detect grow anywhere in method
        if isinstance(inner_node, javalang.tree.MethodInvocation) and inner_node.member == "grow":
            grow_called = True

        # ---------- grow checks ----------
        if isinstance(inner_node, javalang.tree.IfStatement):
            condition = inner_node.condition

            if isinstance(condition, javalang.tree.BinaryOperation):
                left = condition.operandl
                right = condition.operandr
                operator = condition.operator

                then_stmt = inner_node.then_statement
                found_grow_in_if = False

                if then_stmt is not None:
                    for _, then_node in then_stmt:
                        if (
                            isinstance(then_node, javalang.tree.MethodInvocation)
                            and then_node.member == "grow"
                        ):
                            found_grow_in_if = True
                            break

                if found_grow_in_if:
                    grow_called = True
                    metadata_parts.append(f"grow condition: {condition}")

                    # size == length  OR  length == size
                    if (
                        operator == "=="
                        and contains_size(left)
                        and contains_length(right)
                    ) or (
                        operator == "=="
                        and contains_length(left)
                        and contains_size(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_equal")

                    # size >= length  OR  length <= size
                    elif (
                        operator == ">="
                        and contains_size(left)
                        and contains_length(right)
                    ) or (
                        operator == "<="
                        and contains_length(left)
                        and contains_size(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_goe")

                    # length < size + 1  OR  size + 1 > length
                    elif (
                        operator == "<"
                        and contains_length(left)
                        and is_size_plus_one(right)
                    ) or (
                        operator == ">"
                        and is_size_plus_one(left)
                        and contains_length(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_plus_one")

                    # some other size/length-based grow condition
                    elif contains_size(condition) and contains_length(condition):
                        grow_called_with_capacity_check = True
                        tags.append("grow_check_other")

                    # length == size()  OR  size() == length
                    elif (
                        operator == "=="
                        and contains_length(left)
                        and contains_size_function_call(right)
                    ) or (
                        operator == "=="
                        and contains_size_function_call(left)
                        and contains_length(right)
                    ):
                        grow_called_with_capacity_check = True
                        tags.append("growth_check_with_size_function")

        # ---------- array insert using size / assignment-based size updates ----------
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference):
                # size = ...
                if left_side.member == "size":
                    size_updated = True

                    if inner_node.type == "=":
                        value = inner_node.value

                        if isinstance(value, javalang.tree.BinaryOperation):
                            if (
                                isinstance(value.operandl, javalang.tree.MemberReference)
                                and value.operandl.member == "size"
                                and isinstance(value.operandr, javalang.tree.Literal)
                                and value.operandr.value == "1"
                                and value.operator == "+"
                            ):
                                standard_increment = True
                                metadata_parts.append(f"size update: {inner_node}")
                            else:
                                nonstandard_size_change = True
                                metadata_parts.append(f"nonstandard size update: {inner_node}")
                        else:
                            nonstandard_size_change = True
                            metadata_parts.append(f"nonstandard size update: {inner_node}")

                    elif inner_node.type == "+=":
                        value = inner_node.value
                        if isinstance(value, javalang.tree.Literal) and value.value == "1":
                            standard_increment = True
                            metadata_parts.append(f"size update: {inner_node}")
                        else:
                            nonstandard_size_change = True
                            metadata_parts.append(f"nonstandard size update: {inner_node}")

                    else:
                        nonstandard_size_change = True
                        metadata_parts.append(f"nonstandard size update: {inner_node}")

                # check data[...] = ... where index uses size
                if left_side.member == "data" and left_side.selectors:
                    for selector in left_side.selectors:
                        if isinstance(selector, javalang.tree.ArraySelector):
                            index = selector.index

                            if contains_size(index):
                                array_insert_uses_size = True
                                metadata_parts.append(f"array insert: {left_side}")

        # ---------- size++ / ++size / size-- / --size ----------
        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                size_updated = True

                if "++" in (inner_node.postfix_operators or []) or "++" in (inner_node.prefix_operators or []):
                    standard_increment = True
                    metadata_parts.append(f"size update: {inner_node}")
                elif "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    nonstandard_size_change = True
                    metadata_parts.append(f"nonstandard size update: {inner_node}")

    # ---------- finalize tags ----------
    if not grow_called:
        tags.append("grow_missing")
    elif grow_called and not grow_called_with_capacity_check:
        tags.append("grow_called_without_capacity_check")

    if array_insert_uses_size:
        tags.append("array_insert_uses_size")

    if standard_increment:
        tags.append("increment_size")
    elif nonstandard_size_change:
        tags.append("size_change_nonstandard")
    elif not size_updated:
        tags.append("size_no_update")
        metadata_parts.append("no size update detected in add method")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="add",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

## AddAll Function
There is a minimalistic version of tagging along with a more thorough option that can be chosen with a flag

In [60]:
def analyze_addAll_min_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    calls_add = False
    size_updated = False
    standard_increment = False
    nonstandard_size_change = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        if isinstance(inner_node, javalang.tree.MethodInvocation):
            if inner_node.member == "add":
                calls_add = True
                metadata_parts.append(f"calls add: {inner_node}")

        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_updated = True
                metadata_parts.append(f"size update: {inner_node}")

                if inner_node.type == "=":
                    value = inner_node.value

                    if isinstance(value, javalang.tree.BinaryOperation):
                        if (
                            isinstance(value.operandl, javalang.tree.MemberReference)
                            and value.operandl.member == "size"
                            and isinstance(value.operandr, javalang.tree.Literal)
                            and value.operandr.value == "1"
                            and value.operator == "+"
                        ):
                            standard_increment = True
                        else:
                            nonstandard_size_change = True
                    else:
                        nonstandard_size_change = True

                elif inner_node.type == "+=":
                    value = inner_node.value

                    if isinstance(value, javalang.tree.Literal) and value.value == "1":
                        standard_increment = True
                    else:
                        nonstandard_size_change = True

                else:
                    nonstandard_size_change = True

        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                size_updated = True
                metadata_parts.append(f"size update: {inner_node}")

                if "++" in (inner_node.postfix_operators or []) or "++" in (inner_node.prefix_operators or []):
                    standard_increment = True
                elif "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    nonstandard_size_change = True

    if calls_add:
        tags.append("calls_add")
    else:
        tags.append("does_not_call_add")

    if standard_increment:
        tags.append("increment_size")
    elif nonstandard_size_change:
        tags.append("size_change_nonstandard")
    elif not size_updated:
        tags.append("size_no_update")
        metadata_parts.append("no direct size update detected in addAll")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="addAll",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

def analyze_addAll_extended_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    calls_add = False
    size_updated = False
    standard_increment = False
    nonstandard_size_change = False
    batch_size_update = False
    per_element_size_update = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        if isinstance(inner_node, javalang.tree.MethodInvocation):
            if inner_node.member == "add":
                calls_add = True
                metadata_parts.append(f"calls add: {inner_node}")

        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_updated = True
                metadata_parts.append(f"size update: {inner_node}")

                if inner_node.type == "=":
                    value = inner_node.value

                    if isinstance(value, javalang.tree.BinaryOperation):
                        if (
                            isinstance(value.operandl, javalang.tree.MemberReference)
                            and value.operandl.member == "size"
                            and isinstance(value.operandr, javalang.tree.Literal)
                            and value.operandr.value == "1"
                            and value.operator == "+"
                        ):
                            standard_increment = True
                        elif (
                            isinstance(value.operandl, javalang.tree.MemberReference)
                            and value.operandl.member == "size"
                            and isinstance(value.operandr, javalang.tree.MemberReference)
                            and value.operator == "+"
                        ):
                            batch_size_update = True
                        else:
                            nonstandard_size_change = True
                    else:
                        nonstandard_size_change = True

                elif inner_node.type == "+=":
                    value = inner_node.value

                    if isinstance(value, javalang.tree.Literal) and value.value == "1":
                        standard_increment = True
                    elif isinstance(value, javalang.tree.MemberReference):
                        batch_size_update = True
                    else:
                        nonstandard_size_change = True

                else:
                    nonstandard_size_change = True

                parent_types = [type(p).__name__ for p in path if hasattr(p, "__class__")]
                if any(p in parent_types for p in ["ForStatement", "WhileStatement", "EnhancedForControl"]):
                    per_element_size_update = True

        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                size_updated = True
                metadata_parts.append(f"size update: {inner_node}")

                if "++" in (inner_node.postfix_operators or []) or "++" in (inner_node.prefix_operators or []):
                    standard_increment = True

                    parent_types = [type(p).__name__ for p in path if hasattr(p, "__class__")]
                    if any(p in parent_types for p in ["ForStatement", "WhileStatement", "EnhancedForControl"]):
                        per_element_size_update = True

                elif "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    nonstandard_size_change = True

    if calls_add:
        tags.append("calls_add")
    else:
        tags.append("does_not_call_add")

    if standard_increment:
        tags.append("increment_size")
    elif nonstandard_size_change:
        tags.append("size_change_nonstandard")
    elif not size_updated and not batch_size_update:
        tags.append("size_no_update")
        metadata_parts.append("no direct size update detected in addAll")

    if size_updated or batch_size_update:
        tags.append("updates_size_directly")

    if batch_size_update:
        tags.append("batch_size_update")

    if per_element_size_update:
        tags.append("per_element_size_update")

    if calls_add and (size_updated or batch_size_update):
        tags.append("calls_add_and_updates_size")

    if calls_add and not (size_updated or batch_size_update):
        tags.append("calls_add_only")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="addAll",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

## Clear Function 

In [61]:
def analyze_clear_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    assigned_size_to_zero = False
    decrement_size_in_loop = False
    size_updated = False
    nonstandard_size_reset = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # size = ...
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_updated = True
                metadata_parts.append(f"size update: {inner_node}")

                if inner_node.type == "=":
                    value = inner_node.value

                    if isinstance(value, javalang.tree.Literal) and value.value == "0":
                        assigned_size_to_zero = True
                    else:
                        nonstandard_size_reset = True

                else:
                    nonstandard_size_reset = True

        # size-- / --size
        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                if "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    size_updated = True
                    metadata_parts.append(f"size decrement: {inner_node}")

                    parent_types = [type(p).__name__ for p in path if hasattr(p, "__class__")]
                    if any(p in parent_types for p in ["ForStatement", "WhileStatement"]):
                        decrement_size_in_loop = True
                    else:
                        nonstandard_size_reset = True

                elif "++" in (inner_node.postfix_operators or []) or "++" in (inner_node.prefix_operators or []):
                    size_updated = True
                    nonstandard_size_reset = True
                    metadata_parts.append(f"nonstandard size update: {inner_node}")

    if assigned_size_to_zero:
        tags.append("assign_size_to_zero")

    if decrement_size_in_loop:
        tags.append("decrement_size")

    if nonstandard_size_reset and not assigned_size_to_zero and not decrement_size_in_loop:
        tags.append("size_reset_nonstandard")

    if not size_updated:
        tags.append("size_not_reset")
        metadata_parts.append("no size reset detected in clear method")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="clear",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

## Contains & ContainsAll

In [62]:
def analyze_contains_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    loop_bound_size = False
    loop_bound_data_length = False
    loop_bound_other = False
    size_mutated = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # -------- loop bound checks --------
        if isinstance(inner_node, javalang.tree.ForStatement):
            control = inner_node.control

            if isinstance(control, javalang.tree.ForControl):
                condition = control.condition

                if condition is not None:
                    metadata_parts.append(f"for loop condition: {condition}")

                    if contains_size(condition):
                        loop_bound_size = True

                    if contains_length(condition):
                        loop_bound_data_length = True

                    if not contains_size(condition) and not contains_length(condition):
                        loop_bound_other = True

            else:
                loop_bound_other = True
                metadata_parts.append(f"for loop control: {control}")

        elif isinstance(inner_node, javalang.tree.WhileStatement):
            condition = inner_node.condition
            metadata_parts.append(f"while loop condition: {condition}")

            if contains_size(condition):
                loop_bound_size = True

            if contains_length(condition):
                loop_bound_data_length = True

            if not contains_size(condition) and not contains_length(condition):
                loop_bound_other = True

        elif isinstance(inner_node, javalang.tree.EnhancedForControl):
            loop_bound_other = True
            metadata_parts.append(f"enhanced for iterable: {inner_node.iterable}")

        # -------- size mutation checks --------
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_mutated = True
                metadata_parts.append(f"size mutation: {inner_node}")

        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                if (
                    "++" in (inner_node.postfix_operators or [])
                    or "++" in (inner_node.prefix_operators or [])
                    or "--" in (inner_node.postfix_operators or [])
                    or "--" in (inner_node.prefix_operators or [])
                ):
                    size_mutated = True
                    metadata_parts.append(f"size mutation: {inner_node}")

    if loop_bound_size:
        tags.append("loop_bound_size")

    if loop_bound_data_length:
        tags.append("loop_bound_data_length")

    if loop_bound_other:
        tags.append("loop_bound_other")

    if size_mutated:
        tags.append("size_mutated_in_contains")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="contains",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

def analyze_containsAll_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    calls_contains = False
    loop_enhanced_for = False
    loop_standard_for = False
    loop_over_input_collection = False
    loop_other = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # contains(...) call
        if isinstance(inner_node, javalang.tree.MethodInvocation):
            if inner_node.member == "contains":
                calls_contains = True
                metadata_parts.append(f"calls contains: {inner_node}")

        # standard for loop
        if isinstance(inner_node, javalang.tree.ForStatement):
            control = inner_node.control

            if isinstance(control, javalang.tree.ForControl):
                loop_standard_for = True
                condition = control.condition
                metadata_parts.append(f"standard for condition: {condition}")

                cond_text = str(condition) if condition is not None else ""
                if "arg0" in cond_text:
                    loop_over_input_collection = True
                elif "this" not in cond_text and "data" not in cond_text and "size" not in cond_text:
                    loop_other = True

            elif isinstance(control, javalang.tree.EnhancedForControl):
                loop_enhanced_for = True
                iterable = control.iterable
                metadata_parts.append(f"enhanced for iterable: {iterable}")

                iterable_text = str(iterable)
                if "arg0" in iterable_text:
                    loop_over_input_collection = True
                elif "this" not in iterable_text and "data" not in iterable_text:
                    loop_other = True

            else:
                loop_other = True
                metadata_parts.append(f"for loop control: {control}")

        elif isinstance(inner_node, javalang.tree.EnhancedForControl):
            loop_enhanced_for = True
            iterable = inner_node.iterable
            metadata_parts.append(f"enhanced for iterable: {iterable}")

            iterable_text = str(iterable)
            if "arg0" in iterable_text:
                loop_over_input_collection = True
            elif "this" not in iterable_text and "data" not in iterable_text:
                loop_other = True

    if calls_contains:
        tags.append("calls_contains")
    else:
        tags.append("does_not_call_contains")

    if loop_enhanced_for:
        tags.append("loop_enhanced_for")

    if loop_standard_for:
        tags.append("loop_standard_for")

    if loop_over_input_collection:
        tags.append("loop_over_input_collection")

    if loop_other:
        tags.append("loop_other")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="containsAll",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

## isEmpty and Iterator


In [63]:
def analyze_isEmpty_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    size_check = False
    data_length_check = False
    loop_bound_size = False
    loop_bound_data_length = False
    loop_bound_other = False
    has_loop = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # direct emptiness checks
        if isinstance(inner_node, javalang.tree.BinaryOperation):
            left = inner_node.operandl
            right = inner_node.operandr
            operator = inner_node.operator

            if operator == "==" or operator == "!=":
                # size compared to 0
                if (
                    isinstance(left, javalang.tree.MemberReference)
                    and left.member == "size"
                    and isinstance(right, javalang.tree.Literal)
                    and right.value == "0"
                ) or (
                    isinstance(right, javalang.tree.MemberReference)
                    and right.member == "size"
                    and isinstance(left, javalang.tree.Literal)
                    and left.value == "0"
                ):
                    size_check = True
                    metadata_parts.append(f"size check: {inner_node}")

                # data.length compared to 0
                if (
                    isinstance(left, javalang.tree.MemberReference)
                    and left.member == "length"
                    and isinstance(right, javalang.tree.Literal)
                    and right.value == "0"
                ) or (
                    isinstance(right, javalang.tree.MemberReference)
                    and right.member == "length"
                    and isinstance(left, javalang.tree.Literal)
                    and left.value == "0"
                ):
                    data_length_check = True
                    metadata_parts.append(f"data.length check: {inner_node}")

        # loop checks
        if isinstance(inner_node, javalang.tree.ForStatement):
            has_loop = True
            control = inner_node.control

            if isinstance(control, javalang.tree.ForControl):
                condition = control.condition
                metadata_parts.append(f"for loop condition: {condition}")

                if condition is not None:
                    if contains_size(condition):
                        loop_bound_size = True
                    if contains_length(condition):
                        loop_bound_data_length = True
                    if not contains_size(condition) and not contains_length(condition):
                        loop_bound_other = True
                else:
                    loop_bound_other = True

            elif isinstance(control, javalang.tree.EnhancedForControl):
                iterable = control.iterable
                metadata_parts.append(f"enhanced for iterable: {iterable}")
                loop_bound_other = True

            else:
                loop_bound_other = True
                metadata_parts.append(f"for loop control: {control}")

        elif isinstance(inner_node, javalang.tree.WhileStatement):
            has_loop = True
            condition = inner_node.condition
            metadata_parts.append(f"while loop condition: {condition}")

            if contains_size(condition):
                loop_bound_size = True
            if contains_length(condition):
                loop_bound_data_length = True
            if not contains_size(condition) and not contains_length(condition):
                loop_bound_other = True

        elif isinstance(inner_node, javalang.tree.EnhancedForControl):
            has_loop = True
            iterable = inner_node.iterable
            metadata_parts.append(f"enhanced for iterable: {iterable}")
            loop_bound_other = True

    if size_check:
        tags.append("size_check")

    if data_length_check:
        tags.append("data_length_check")

    if loop_bound_size:
        tags.append("loop_bound_size")

    if loop_bound_data_length:
        tags.append("loop_bound_data_length")

    if loop_bound_other:
        tags.append("loop_bound_other")

    if not has_loop:
        tags.append("no_loop")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="isEmpty",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

def analyze_iterator_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    returns_new_arraycollectioniterator = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        if isinstance(inner_node, javalang.tree.ReturnStatement):
            expr = inner_node.expression

            if isinstance(expr, javalang.tree.ClassCreator):
                if expr.type is not None and expr.type.name == "ArrayCollectionIterator":
                    returns_new_arraycollectioniterator = True
                    metadata_parts.append(f"return statement: {inner_node}")

    if returns_new_arraycollectioniterator:
        tags.append("return_new_arraycollectioniterator")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="iterator",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

## Remove & RemoveAll Functions

In [64]:
def analyze_remove_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    loop_count = 0
    loop_bound_size = False
    loop_bound_data_length = False
    loop_bound_other = False

    uses_contains = False
    iterator_removal = False
    array_reconstruction = False
    item_removed = False

    decrement_size = False
    size_decrement_seen = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # ---------- loop detection ----------
        if isinstance(inner_node, javalang.tree.ForStatement):
            loop_count += 1
            control = inner_node.control

            if isinstance(control, javalang.tree.ForControl):
                condition = control.condition
                metadata_parts.append(f"for loop condition: {condition}")

                if condition is not None:
                    if contains_size(condition):
                        loop_bound_size = True
                    if contains_length(condition):
                        loop_bound_data_length = True
                    if not contains_size(condition) and not contains_length(condition):
                        loop_bound_other = True
                else:
                    loop_bound_other = True

            elif isinstance(control, javalang.tree.EnhancedForControl):
                metadata_parts.append(f"enhanced for iterable: {control.iterable}")
                if contains_size(control.iterable):
                    loop_bound_size = True
                elif contains_length(control.iterable):
                    loop_bound_data_length = True
                else:
                    loop_bound_other = True

        elif isinstance(inner_node, javalang.tree.WhileStatement):
            loop_count += 1
            condition = inner_node.condition
            metadata_parts.append(f"while loop condition: {condition}")

            if contains_size(condition):
                loop_bound_size = True
            if contains_length(condition):
                loop_bound_data_length = True
            if not contains_size(condition) and not contains_length(condition):
                loop_bound_other = True

        # ---------- contains() usage ----------
        if isinstance(inner_node, javalang.tree.MethodInvocation):
            if inner_node.member == "contains":
                uses_contains = True
                metadata_parts.append(f"uses contains: {inner_node}")

            if inner_node.member == "remove":
                qualifier_text = str(getattr(inner_node, "qualifier", ""))
                if "iter" in qualifier_text.lower() or "iterator" in qualifier_text.lower():
                    iterator_removal = True
                    item_removed = True
                    metadata_parts.append(f"iterator removal: {inner_node}")

            if inner_node.member == "arraycopy":
                item_removed = True
                metadata_parts.append(f"array copy removal/coalesce: {inner_node}")

        # ---------- array reconstruction ----------
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl
            right_side = inner_node.value

            # size decrement patterns
            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_decrement_seen = True
                metadata_parts.append(f"size update: {inner_node}")

                if inner_node.type == "=" and isinstance(right_side, javalang.tree.BinaryOperation):
                    if (
                        isinstance(right_side.operandl, javalang.tree.MemberReference)
                        and right_side.operandl.member == "size"
                        and isinstance(right_side.operandr, javalang.tree.Literal)
                        and right_side.operandr.value == "1"
                        and right_side.operator == "-"
                    ):
                        decrement_size = True
                    else:
                        metadata_parts.append(f"nonstandard size update: {inner_node}")

                elif inner_node.type == "-=":
                    if isinstance(right_side, javalang.tree.Literal) and right_side.value == "1":
                        decrement_size = True
                    else:
                        metadata_parts.append(f"nonstandard size update: {inner_node}")

            # detect data[i] = data[i+1] style coalescing
            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "data":
                if left_side.selectors:
                    item_removed = True
                    metadata_parts.append(f"data assignment during remove: {inner_node}")

            # detect new array assigned into data
            if (
                isinstance(left_side, javalang.tree.MemberReference)
                and left_side.member == "data"
            ):
                if isinstance(right_side, (javalang.tree.ArrayCreator, javalang.tree.Cast)):
                    array_reconstruction = True
                    item_removed = True
                    metadata_parts.append(f"array reconstruction: {inner_node}")

        # ---------- size-- / --size ----------
        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                if "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    size_decrement_seen = True
                    decrement_size = True
                    metadata_parts.append(f"size decrement: {inner_node}")

    # ---------- finalize tags ----------
    if item_removed:
        tags.append("item_removed")

    if item_removed and decrement_size:
        tags.append("decrement_size")

    if loop_bound_size:
        tags.append("loop_bound_size")

    if loop_bound_data_length:
        tags.append("loop_bound_data_length")

    if loop_bound_other:
        tags.append("loop_bound_other")

    if loop_count == 1:
        tags.append("one_loop_solution")
    elif loop_count >= 2:
        tags.append("two_loop_solution")

    if uses_contains:
        tags.append("uses_contains")

    if array_reconstruction:
        tags.append("array_reconstruction")

    if iterator_removal:
        tags.append("iterator_removal")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="remove",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

def analyze_removeAll_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    calls_contains = False
    calls_remove = False

    size_updated = False
    decrements_size = False
    size_change_nonstandard = False

    loop_over_input_collection = False
    loop_over_self_collection = False
    loop_other = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # method calls
        if isinstance(inner_node, javalang.tree.MethodInvocation):
            if inner_node.member == "contains":
                calls_contains = True
                metadata_parts.append(f"calls contains: {inner_node}")

            if inner_node.member == "remove":
                calls_remove = True
                metadata_parts.append(f"calls remove: {inner_node}")

        # loop structure
        if isinstance(inner_node, javalang.tree.ForStatement):
            control = inner_node.control

            if isinstance(control, javalang.tree.EnhancedForControl):
                iterable = control.iterable
                iterable_text = str(iterable)
                metadata_parts.append(f"enhanced for iterable: {iterable}")

                if "arg0" in iterable_text:
                    loop_over_input_collection = True
                elif "this" in iterable_text or "data" in iterable_text:
                    loop_over_self_collection = True
                else:
                    loop_other = True

            elif isinstance(control, javalang.tree.ForControl):
                condition = control.condition
                cond_text = str(condition) if condition is not None else ""
                metadata_parts.append(f"standard for condition: {condition}")

                if "arg0" in cond_text:
                    loop_over_input_collection = True
                elif "this" in cond_text or "data" in cond_text or "size" in cond_text:
                    loop_over_self_collection = True
                else:
                    loop_other = True
            else:
                loop_other = True
                metadata_parts.append(f"for loop control: {control}")

        elif isinstance(inner_node, javalang.tree.WhileStatement):
            condition = inner_node.condition
            cond_text = str(condition)
            metadata_parts.append(f"while loop condition: {condition}")

            if "arg0" in cond_text:
                loop_over_input_collection = True
            elif "this" in cond_text or "data" in cond_text or "size" in cond_text:
                loop_over_self_collection = True
            else:
                loop_other = True

        elif isinstance(inner_node, javalang.tree.EnhancedForControl):
            iterable = inner_node.iterable
            iterable_text = str(iterable)
            metadata_parts.append(f"enhanced for iterable: {iterable}")

            if "arg0" in iterable_text:
                loop_over_input_collection = True
            elif "this" in iterable_text or "data" in iterable_text:
                loop_over_self_collection = True
            else:
                loop_other = True

        # size assignment updates
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_updated = True
                metadata_parts.append(f"size update: {inner_node}")

                if inner_node.type == "=":
                    value = inner_node.value
                    if isinstance(value, javalang.tree.BinaryOperation):
                        if (
                            isinstance(value.operandl, javalang.tree.MemberReference)
                            and value.operandl.member == "size"
                            and isinstance(value.operandr, javalang.tree.Literal)
                            and value.operandr.value == "1"
                            and value.operator == "-"
                        ):
                            decrements_size = True
                        else:
                            size_change_nonstandard = True
                    else:
                        size_change_nonstandard = True

                elif inner_node.type == "-=":
                    value = inner_node.value
                    if isinstance(value, javalang.tree.Literal) and value.value == "1":
                        decrements_size = True
                    else:
                        size_change_nonstandard = True
                else:
                    size_change_nonstandard = True

        # size-- / --size
        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                if "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    size_updated = True
                    decrements_size = True
                    metadata_parts.append(f"size decrement: {inner_node}")
                elif "++" in (inner_node.postfix_operators or []) or "++" in (inner_node.prefix_operators or []):
                    size_updated = True
                    size_change_nonstandard = True
                    metadata_parts.append(f"nonstandard size update: {inner_node}")

    # finalize tags
    if calls_contains:
        tags.append("calls_contains")
    else:
        tags.append("does_not_call_contains")

    if calls_remove:
        tags.append("calls_remove")
    else:
        tags.append("does_not_call_remove")

    if loop_over_input_collection:
        tags.append("loop_over_input_collection")

    if loop_over_self_collection:
        tags.append("loop_over_self_collection")

    if loop_other:
        tags.append("loop_other")

    if calls_remove and not size_updated:
        tags.append("calls_remove_only")

    if calls_remove and decrements_size:
        tags.append("calls_remove_and_decrements_size")

    if not calls_remove and decrements_size:
        tags.append("decrements_size")

    if size_change_nonstandard:
        tags.append("size_change_nonstandard")

    if not calls_remove and not size_updated:
        tags.append("size_no_update")
        metadata_parts.append("no direct size update detected in removeAll")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="removeAll",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

## RetainAll, toArray, toSortedList

In [65]:
def analyze_retainAll_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    calls_contains = False
    calls_remove = False
    iterator_removal = False

    size_updated = False
    decrements_size = False
    size_change_nonstandard = False

    loop_over_self_collection = False
    loop_over_input_collection = False
    loop_other = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # method calls
        if isinstance(inner_node, javalang.tree.MethodInvocation):
            if inner_node.member == "contains":
                calls_contains = True
                metadata_parts.append(f"calls contains: {inner_node}")

            if inner_node.member == "remove":
                calls_remove = True
                metadata_parts.append(f"calls remove: {inner_node}")

                qualifier_text = str(getattr(inner_node, "qualifier", ""))
                if "iter" in qualifier_text.lower() or "iterator" in qualifier_text.lower():
                    iterator_removal = True
                    metadata_parts.append(f"iterator removal: {inner_node}")

        # loop structure
        if isinstance(inner_node, javalang.tree.ForStatement):
            control = inner_node.control

            if isinstance(control, javalang.tree.EnhancedForControl):
                iterable = control.iterable
                iterable_text = str(iterable)
                metadata_parts.append(f"enhanced for iterable: {iterable}")

                if "this" in iterable_text or "data" in iterable_text:
                    loop_over_self_collection = True
                elif "arg0" in iterable_text:
                    loop_over_input_collection = True
                else:
                    loop_other = True

            elif isinstance(control, javalang.tree.ForControl):
                condition = control.condition
                cond_text = str(condition) if condition is not None else ""
                metadata_parts.append(f"standard for condition: {condition}")

                if "this" in cond_text or "data" in cond_text or "size" in cond_text:
                    loop_over_self_collection = True
                elif "arg0" in cond_text:
                    loop_over_input_collection = True
                else:
                    loop_other = True

            else:
                loop_other = True
                metadata_parts.append(f"for loop control: {control}")

        elif isinstance(inner_node, javalang.tree.WhileStatement):
            condition = inner_node.condition
            cond_text = str(condition)
            metadata_parts.append(f"while loop condition: {condition}")

            if "this" in cond_text or "data" in cond_text or "size" in cond_text:
                loop_over_self_collection = True
            elif "arg0" in cond_text:
                loop_over_input_collection = True
            else:
                loop_other = True

        elif isinstance(inner_node, javalang.tree.EnhancedForControl):
            iterable = inner_node.iterable
            iterable_text = str(iterable)
            metadata_parts.append(f"enhanced for iterable: {iterable}")

            if "this" in iterable_text or "data" in iterable_text:
                loop_over_self_collection = True
            elif "arg0" in iterable_text:
                loop_over_input_collection = True
            else:
                loop_other = True

        # size assignment updates
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl

            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_updated = True
                metadata_parts.append(f"size update: {inner_node}")

                if inner_node.type == "=":
                    value = inner_node.value
                    if isinstance(value, javalang.tree.BinaryOperation):
                        if (
                            isinstance(value.operandl, javalang.tree.MemberReference)
                            and value.operandl.member == "size"
                            and isinstance(value.operandr, javalang.tree.Literal)
                            and value.operandr.value == "1"
                            and value.operator == "-"
                        ):
                            decrements_size = True
                        else:
                            size_change_nonstandard = True
                    else:
                        size_change_nonstandard = True

                elif inner_node.type == "-=":
                    value = inner_node.value
                    if isinstance(value, javalang.tree.Literal) and value.value == "1":
                        decrements_size = True
                    else:
                        size_change_nonstandard = True
                else:
                    size_change_nonstandard = True

        # size-- / --size
        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                if "--" in (inner_node.postfix_operators or []) or "--" in (inner_node.prefix_operators or []):
                    size_updated = True
                    decrements_size = True
                    metadata_parts.append(f"size decrement: {inner_node}")
                elif "++" in (inner_node.postfix_operators or []) or "++" in (inner_node.prefix_operators or []):
                    size_updated = True
                    size_change_nonstandard = True
                    metadata_parts.append(f"nonstandard size update: {inner_node}")

    # finalize tags
    if calls_contains:
        tags.append("calls_contains")
    else:
        tags.append("does_not_call_contains")

    if calls_remove:
        tags.append("calls_remove")
    else:
        tags.append("does_not_call_remove")

    if iterator_removal:
        tags.append("iterator_removal")

    if loop_over_self_collection:
        tags.append("loop_over_self_collection")

    if loop_over_input_collection:
        tags.append("loop_over_input_collection")

    if loop_other:
        tags.append("loop_other")

    if calls_remove and not size_updated:
        tags.append("calls_remove_only")

    if calls_remove and decrements_size:
        tags.append("calls_remove_and_decrements_size")

    if not calls_remove and decrements_size:
        tags.append("decrements_size")

    if size_change_nonstandard:
        tags.append("size_change_nonstandard")

    if not calls_remove and not size_updated:
        tags.append("size_no_update")
        metadata_parts.append("no direct size update detected in retainAll")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="retainAll",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

def analyze_toArray_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    returns_new_array = False
    uses_loop_copy = False
    loop_bound_size = False
    loop_bound_data_length = False
    loop_bound_other = False
    has_loop = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # detect new array creation / return
        if isinstance(inner_node, javalang.tree.ArrayCreator):
            returns_new_array = True
            metadata_parts.append(f"new array created: {inner_node}")

        if isinstance(inner_node, javalang.tree.ReturnStatement):
            metadata_parts.append(f"return statement: {inner_node}")

        # loop checks
        if isinstance(inner_node, javalang.tree.ForStatement):
            has_loop = True
            uses_loop_copy = True
            control = inner_node.control

            if isinstance(control, javalang.tree.ForControl):
                condition = control.condition
                metadata_parts.append(f"for loop condition: {condition}")

                if condition is not None:
                    if contains_size(condition):
                        loop_bound_size = True
                    if contains_length(condition):
                        loop_bound_data_length = True
                    if not contains_size(condition) and not contains_length(condition):
                        loop_bound_other = True
                else:
                    loop_bound_other = True

            elif isinstance(control, javalang.tree.EnhancedForControl):
                metadata_parts.append(f"enhanced for iterable: {control.iterable}")
                loop_bound_other = True

            else:
                loop_bound_other = True
                metadata_parts.append(f"for loop control: {control}")

        elif isinstance(inner_node, javalang.tree.WhileStatement):
            has_loop = True
            uses_loop_copy = True
            condition = inner_node.condition
            metadata_parts.append(f"while loop condition: {condition}")

            if contains_size(condition):
                loop_bound_size = True
            if contains_length(condition):
                loop_bound_data_length = True
            if not contains_size(condition) and not contains_length(condition):
                loop_bound_other = True

        elif isinstance(inner_node, javalang.tree.EnhancedForControl):
            has_loop = True
            uses_loop_copy = True
            metadata_parts.append(f"enhanced for iterable: {inner_node.iterable}")
            loop_bound_other = True

    if returns_new_array:
        tags.append("returns_new_array")

    if uses_loop_copy:
        tags.append("uses_loop_copy")

    if loop_bound_size:
        tags.append("loop_bound_size")

    if loop_bound_data_length:
        tags.append("loop_bound_data_length")

    if loop_bound_other:
        tags.append("loop_bound_other")

    if not has_loop:
        tags.append("no_loop")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="toArray",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )

def analyze_toSortedList_function_node(node, file_name):
    tags = []
    line_numbers = []
    metadata_parts = []

    if node.position is not None:
        line_numbers.append(node.position.line)

    loop_bound_size = False
    loop_bound_data_length = False
    loop_bound_other = False

    copies_elements_using_size = False
    copies_elements_using_data_length = False
    uses_size_to_limit_sort_scope = False

    size_mutated = False

    for path, inner_node in node:
        if getattr(inner_node, "position", None) is not None:
            line_numbers.append(inner_node.position.line)

        # loop analysis
        if isinstance(inner_node, javalang.tree.ForStatement):
            control = inner_node.control

            if isinstance(control, javalang.tree.ForControl):
                condition = control.condition
                metadata_parts.append(f"for loop condition: {condition}")

                if condition is not None:
                    if contains_size(condition):
                        loop_bound_size = True
                        uses_size_to_limit_sort_scope = True

                    if contains_length(condition):
                        loop_bound_data_length = True

                    if not contains_size(condition) and not contains_length(condition):
                        loop_bound_other = True
            else:
                loop_bound_other = True
                metadata_parts.append(f"for loop control: {control}")

        elif isinstance(inner_node, javalang.tree.WhileStatement):
            condition = inner_node.condition
            metadata_parts.append(f"while loop condition: {condition}")

            if contains_size(condition):
                loop_bound_size = True
                uses_size_to_limit_sort_scope = True

            if contains_length(condition):
                loop_bound_data_length = True

            if not contains_size(condition) and not contains_length(condition):
                loop_bound_other = True

        # copy behavior: add(...) into sortable list using data[i] style access
        if isinstance(inner_node, javalang.tree.MethodInvocation):
            if inner_node.member == "add":
                metadata_parts.append(f"add call: {inner_node}")

                for arg in inner_node.arguments or []:
                    arg_text = str(arg)

                    if "data" in arg_text and "size" in arg_text:
                        copies_elements_using_size = True
                    elif "data" in arg_text and "length" in arg_text:
                        copies_elements_using_data_length = True

        # assignment-based copy behavior
        if isinstance(inner_node, javalang.tree.Assignment):
            left_side = inner_node.expressionl
            right_side = inner_node.value
            assign_text = str(inner_node)

            if "data" in assign_text:
                if "size" in assign_text:
                    copies_elements_using_size = True
                if "length" in assign_text:
                    copies_elements_using_data_length = True

            # detect size mutation
            if isinstance(left_side, javalang.tree.MemberReference) and left_side.member == "size":
                size_mutated = True
                metadata_parts.append(f"size mutation: {inner_node}")

        # size++ / --size etc.
        if isinstance(inner_node, javalang.tree.MemberReference) and inner_node.member == "size":
            if inner_node.postfix_operators or inner_node.prefix_operators:
                if (
                    "++" in (inner_node.postfix_operators or [])
                    or "++" in (inner_node.prefix_operators or [])
                    or "--" in (inner_node.postfix_operators or [])
                    or "--" in (inner_node.prefix_operators or [])
                ):
                    size_mutated = True
                    metadata_parts.append(f"size mutation: {inner_node}")

    if loop_bound_size:
        tags.append("loop_bound_size")

    if loop_bound_data_length:
        tags.append("loop_bound_data_length")

    if loop_bound_other:
        tags.append("loop_bound_other")

    if uses_size_to_limit_sort_scope:
        tags.append("uses_size_to_limit_sort_scope")
    else:
        tags.append("ignores_size_in_sort_scope")

    if copies_elements_using_size:
        tags.append("copies_elements_using_size")

    if copies_elements_using_data_length:
        tags.append("copies_elements_using_data_length")

    if size_mutated:
        tags.append("size_mutated_in_toSortedList")
    else:
        tags.append("size_not_mutated_in_toSortedList")

    if len(tags) == 0:
        tags.append("NULL")

    return AnalysisRow(
        file_name=file_name,
        class_name="ArrayCollection",
        method_name="toSortedList",
        kind="method",
        line_numbers=sorted(set(line_numbers)),
        tags=sorted(set(tags)),
        metadata=" | ".join(metadata_parts) if metadata_parts else str(node)
    )


## Main Function to run AST Node Analysis to produce List of AnalysisRows

In [66]:
def process_node(node, file_name):
    # size()
    if (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "size"
        and len(node.parameters) == 0
    ):
        return analyze_size_function_node(node, file_name)

    # ArrayCollection()
    elif (
        isinstance(node, javalang.tree.ConstructorDeclaration)
        and node.name == "ArrayCollection"
        and len(node.parameters) == 0
    ):
        return analyze_constructor_function_node(node, file_name)

    # add(T arg0)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "add"
        and len(node.parameters) == 1
    ):
        return analyze_add_function_node(node, file_name)

    # addAll(Collection<? extends T> arg0)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "addAll"
        and len(node.parameters) == 1
    ):
        # return analyze_addAll_min_function_node(node, file_name)
        return analyze_addAll_extended_function_node(node, file_name)

    # clear()
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "clear"
        and len(node.parameters) == 0
    ):
        return analyze_clear_function_node(node, file_name)

    # contains(Object arg0)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "contains"
        and len(node.parameters) == 1
    ):
        return analyze_contains_function_node(node, file_name)

    # containsAll(Collection<?> arg0)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "containsAll"
        and len(node.parameters) == 1
    ):
        return analyze_containsAll_function_node(node, file_name)

    # isEmpty()
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "isEmpty"
        and len(node.parameters) == 0
    ):
        return analyze_isEmpty_function_node(node, file_name)

    # iterator()
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "iterator"
        and len(node.parameters) == 0
    ):
        return analyze_iterator_function_node(node, file_name)

    # remove(Object arg0)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "remove"
        and len(node.parameters) == 1
    ):
        return analyze_remove_function_node(node, file_name)

    # removeAll(Collection<?> arg0)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "removeAll"
        and len(node.parameters) == 1
    ):
        return analyze_removeAll_function_node(node, file_name)

    # retainAll(Collection<?> arg0)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "retainAll"
        and len(node.parameters) == 1
    ):
        return analyze_retainAll_function_node(node, file_name)

    # toArray()
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "toArray"
        and len(node.parameters) == 0
    ):
        return analyze_toArray_function_node(node, file_name)

    # toSortedList(Comparator<? super T> cmp)
    elif (
        isinstance(node, javalang.tree.MethodDeclaration)
        and node.name == "toSortedList"
        and len(node.parameters) == 1
    ):
        return analyze_toSortedList_function_node(node, file_name)

    return None


def analyze_all_submissions(raw_trees):
    all_rows = []

    for file_name, tree in raw_trees.items():
        for path, node in tree:
            result = process_node(node, file_name)
            if result is not None:
                all_rows.append(result)

    return all_rows


info = analyze_all_submissions(raw_trees)
print(info)

[AnalysisRow(file_name='AfricanPenguin.java', class_name='ArrayCollection', method_name='ArrayCollection', kind='constructor', line_numbers=[22], tags=['sets_size_to_0'], metadata='ConstructorDeclaration(annotations=[Annotation(element=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value="unchecked"), name=SuppressWarnings)], body=[StatementExpression(expression=Assignment(expressionl=MemberReference(member=size, postfix_operators=[], prefix_operators=[], qualifier=, selectors=[]), type==, value=Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value=0)), label=None), StatementExpression(expression=Assignment(expressionl=MemberReference(member=data, postfix_operators=[], prefix_operators=[], qualifier=, selectors=[]), type==, value=Cast(expression=ArrayCreator(dimensions=[Literal(postfix_operators=[], prefix_operators=[], qualifier=None, selectors=[], value=10)], initializer=None, postfix_operators=[], prefix_operators=[]

## Converting to a .csv
This will take the finalized listed data structure of AnalysisRow and process the information into a .csv file that is easily readable


In [67]:
import csv

def write_analysis_rows_to_csv(info, output_file="analysis_output.csv"):
    with open(output_file, "w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)

        writer.writerow([
            "File Name",
            "Class Name",
            "Method Name",
            "Kind",
            "Line Numbers",
            "Tags",
            "Metadata"
        ])

        for row in info:
            writer.writerow([
                row.file_name,
                row.class_name,
                row.method_name,
                row.kind,
                ", ".join(str(num) for num in row.line_numbers),
                ", ".join(row.tags),
                row.metadata
            ])

    print(f"CSV written to {output_file}")

write_analysis_rows_to_csv(info)

CSV written to analysis_output.csv
